In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

df = pd.read_json(
    "../data/processed/Movies_Features.json"
)

print(df.shape)
print(df.head())

threshold = df["vote_count"].quantile(0.75)

df["high_engagement"] = (
    df["vote_count"] >= threshold
).astype(int)

print("Engagement threshold:", threshold)
print(df["high_engagement"].value_counts())

features = [
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "num_genres",
    "num_keywords",
    "release_year",
    "release_month",
    "release_decade",
    "profit"
]

X = df[features]
y = df["high_engagement"]

X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

(2740, 20)
   movie_id                      title  \
0    969681  Spider-Man: Brand New Day   
1   1423191              Resident Evil   
2   1101383      The End of Oak Street   
3   1204680            Coyote vs. Acme   
4   1368337                The Odyssey   

                                            overview  release_date  runtime  \
0  Fighting crime full-time as Spider-Man in a wo...     1785283.0      145   
1  Medical courier Bryan unwittingly finds himsel...     1789516.0       95   
2  After a mysterious cosmic event rips Oak Stree...     1786492.0      100   
3  After Acme products fail him one too many time...     1786924.0      103   
4  Odysseus, the legendary King of Ithaca, embark...     1784073.0      173   

  original_language                                genres  \
0                en  [Science Fiction, Action, Adventure]   
1                en  [Horror, Science Fiction, Adventure]   
2                en  [Science Fiction, Mystery, Thriller]   
3                

In [2]:
from sklearn.model_selection import StratifiedKFold

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])

random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

svm_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LinearSVC(
            max_iter=5000,
            random_state=42
        )
    )
])

In [3]:
from sklearn.model_selection import cross_validate


metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

logistic_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=metrics
)

rf_scores = cross_validate(
    random_forest,
    X_train,
    y_train,
    cv=cv,
    scoring=metrics
)

svm_scores = cross_validate(
    svm_model,
    X_train,
    y_train,
    cv=cv,
    scoring=metrics
)